# 02 — Materials Feature Engineering and External Data

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

PROCESSED = ROOT / "data" / "processed"
SOURCE = ROOT / "source_data"
TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
DIAG = ROOT / "results" / "diagnostics"

for p in [PROCESSED, SOURCE, TABLES, FIGURES, DIAG]:
    p.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 21, 63]
PRIMARY_HORIZON = 21
ANN = 252

sector = pd.read_parquet(PROCESSED / "materials_daily_returns.parquet").copy()
sector["Date"] = pd.to_datetime(sector["Date"])
sector = sector.sort_values("Date").reset_index(drop=True)

top50 = pd.read_parquet(PROCESSED / "materials_top50_panel.parquet").copy()
top50["Date"] = pd.to_datetime(top50["Date"])

r = sector.set_index("Date")["Sector_Return"].astype(float)

assert np.isfinite(r).all()
assert sector["N_Constituents"].le(50).all()

print("Sector observations:", len(sector))
print("Top-50 stock-date observations:", len(top50))

Sector observations: 2207
Top-50 stock-date observations: 110350


## 1. Forward realised-volatility targets


In [2]:
target = pd.DataFrame(index=r.index)

for h in HORIZONS:
    future_sq = pd.concat(
        [r.shift(-j).pow(2) for j in range(1, h + 1)],
        axis=1
    )
    target[f"Target_RV_{h}"] = np.sqrt(
        (ANN / h) * future_sq.sum(axis=1, min_count=h)
    )

target = target.reset_index()

# Direct target-alignment audit.
audit = []
for h in HORIZONS:
    col = f"Target_RV_{h}"
    valid = target.dropna(subset=[col])
    step = max(1, len(valid) // 25)
    for dte in valid["Date"].iloc[::step]:
        i = r.index.get_loc(dte)
        direct = np.sqrt((ANN / h) * np.sum(r.iloc[i+1:i+h+1].values ** 2))
        stored = valid.loc[valid["Date"].eq(dte), col].iloc[0]
        audit.append([dte, h, stored, direct, abs(stored - direct)])

audit = pd.DataFrame(
    audit,
    columns=["Date", "Horizon", "Stored", "Direct", "Abs_Difference"]
)

assert audit["Abs_Difference"].max() < 1e-10
audit.to_csv(DIAG / "02_target_alignment_audit.csv", index=False)

display(
    target[[f"Target_RV_{h}" for h in HORIZONS]]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
Target_RV_5,2202.0,0.360702,0.411281,0.030287,0.154902,0.216538,0.331402,2.793305
Target_RV_21,2186.0,0.386137,0.389774,0.084642,0.180301,0.230702,0.338874,2.083992
Target_RV_63,2144.0,0.401113,0.380544,0.136708,0.188755,0.242445,0.337032,1.797901


## 2. F1 volatility and F2 return/downside features


In [3]:
feat = pd.DataFrame(index=r.index)

# F1 — historical volatility state.
for w in [5, 10, 21, 63, 126]:
    feat[f"RV_{w}"] = np.sqrt(
        ANN * r.pow(2).rolling(w).mean()
    )

feat["EWMA_Vol_094"] = np.sqrt(
    ANN * r.pow(2).ewm(alpha=0.06, adjust=False).mean()
)

feat["RV21_Change_1D"] = feat["RV_21"].pct_change()
feat["RV21_Change_5D"] = feat["RV_21"].pct_change(5)
feat["VolOfVol_21"] = feat["RV_21"].rolling(21).std()
feat["VolOfVol_63"] = feat["RV_21"].rolling(63).std()

# F2 — returns, downside risk and drawdown.
feat["Return_1D"] = r
feat["Abs_Return_1D"] = r.abs()
feat["Squared_Return_1D"] = r.pow(2)

for w in [5, 21, 63]:
    feat[f"Return_{w}D"] = (
        (1 + r).rolling(w).apply(np.prod, raw=True) - 1
    )

for w in [21, 63]:
    feat[f"Downside_Vol_{w}"] = np.sqrt(
        ANN * r.clip(upper=0).pow(2).rolling(w).mean()
    )

wealth = (1 + r).cumprod()
for w in [21, 63, 126]:
    feat[f"Drawdown_{w}"] = (
        wealth / wealth.rolling(w).max() - 1
    )

## 3. F3 range and liquidity — same top-50 universe


In [4]:
q = top50.copy()

q["HL_Range"] = np.log(
    q["High"] / q["Low"]
).replace([np.inf, -np.inf], np.nan)

q["Parkinson_Var"] = (
    q["HL_Range"].pow(2) / (4 * np.log(2))
)

q["Dollar_Volume"] = (
    pd.to_numeric(q["Close"], errors="coerce")
    * pd.to_numeric(q["Volume"], errors="coerce")
)

if "Weight" not in q.columns:
    q["Weight"] = q["Lagged_MCap"] / q.groupby("Date")["Lagged_MCap"].transform("sum")

def weighted_daily_mean(frame, value_col):
    z = frame[["Date", "Weight", value_col]].copy()
    z[value_col] = pd.to_numeric(z[value_col], errors="coerce")
    z = z[np.isfinite(z[value_col]) & np.isfinite(z["Weight"]) & z["Weight"].gt(0)]
    z["wx"] = z["Weight"] * z[value_col]
    out = z.groupby("Date").agg(
        numerator=("wx", "sum"),
        denominator=("Weight", "sum")
    )
    return out["numerator"] / out["denominator"]

range_daily = pd.DataFrame(index=r.index)
range_daily["Sector_HL_Range"] = weighted_daily_mean(q, "HL_Range")
range_daily["Sector_Parkinson_Var"] = weighted_daily_mean(q, "Parkinson_Var")
range_daily["Sector_Parkinson_Vol"] = np.sqrt(
    ANN * range_daily["Sector_Parkinson_Var"]
)

dv = (
    q.assign(Dollar_Volume=q["Dollar_Volume"].where(q["Dollar_Volume"].gt(0)))
    .groupby("Date")["Dollar_Volume"]
    .sum(min_count=1)
)
range_daily["Sector_Log_Dollar_Volume"] = np.log(dv)
range_daily["Dollar_Volume_Change_21D"] = dv.pct_change(21)

for c in range_daily.columns:
    feat[c] = range_daily[c]

display(range_daily.describe().T)

,count,mean,std,min,25%,50%,75%,max
Sector_HL_Range,2207.0,0.043138,0.043555,0.011086,0.019124,0.023420,0.043297,0.251183
Sector_Parkinson_Var,2207.0,0.004765,0.010315,0.000084,0.000232,0.000348,0.002401,0.077112
Sector_Parkinson_Vol,2207.0,0.713836,0.831603,0.145219,0.241719,0.295940,0.777898,4.408207
Sector_Log_Dollar_Volume,2207.0,20.946610,0.418055,17.993376,20.657649,20.927633,21.188806,23.624675
Dollar_Volume_Change_21D,2186.0,0.126143,1.069433,-0.931873,-0.205319,0.009550,0.268605,40.254936


## 4. F4 broader market, AUD/USD and commodity variables

In [5]:
market_map = {
    # name       Yahoo ticker   lag one source observation?   timing rationale
    "AUDUSD": ("AUDUSD=X", True,  "Conservative FX availability at ASX close"),
    "VIX":    ("^VIX",     True,  "US session timing"),
    "ASX200": ("^AXJO",    False, "Same Australian close is observable"),
    "GOLD":   ("GC=F",     True,  "Global/US futures timing"),
    "COPPER": ("HG=F",     True,  "Global/US futures timing"),
}

external = {}
acq_rows = []

for name, (ticker, shift_one, timing_rationale) in market_map.items():
    try:
        x = yf.download(
            ticker,
            start="2017-01-01",
            progress=False,
            auto_adjust=False
        )
        if isinstance(x.columns, pd.MultiIndex):
            x.columns = x.columns.get_level_values(0)

        px = x["Adj Close"] if "Adj Close" in x.columns else x["Close"]
        s = pd.Series(
            pd.to_numeric(px, errors="coerce").values,
            index=pd.to_datetime(px.index).tz_localize(None),
            name=name
        ).sort_index()

        if shift_one:
            s = s.shift(1)
        s = s.reindex(feat.index).ffill(limit=3)
        ret = s.pct_change(fill_method=None)
        external[f"{name}_Return_1D"] = ret
        external[f"{name}_Return_21D"] = s.pct_change(21, fill_method=None)
        external[f"{name}_RV21"] = np.sqrt(ANN * ret.pow(2).rolling(21).mean())
        if name == "VIX":
            external["VIX_Level"] = s

        acq_rows.append([name, ticker, shift_one, timing_rationale, "Downloaded"])

    except Exception as e:
        acq_rows.append([name, ticker, shift_one, timing_rationale, f"Unavailable: {type(e).__name__}"])

iron_file = SOURCE / "iron_ore_daily.csv"
if iron_file.exists():
    iron = pd.read_csv(iron_file, parse_dates=["Date"]).sort_values("Date").set_index("Date")
    candidates = [c for c in iron.columns if any(k in c.lower() for k in ["iron", "close", "price"])]
    if not candidates:
        raise KeyError("iron_ore_daily.csv needs a price/close column.")
    s = pd.to_numeric(iron[candidates[0]], errors="coerce")
    s.index = pd.to_datetime(s.index)
    s = s.shift(1).reindex(feat.index).ffill(limit=3)
    ret = s.pct_change(fill_method=None)
    external["IRON_ORE_Return_1D"] = ret
    external["IRON_ORE_Return_21D"] = s.pct_change(21, fill_method=None)
    external["IRON_ORE_RV21"] = np.sqrt(ANN * ret.pow(2).rolling(21).mean())
    acq_rows.append(["IRON_ORE", "source_data/iron_ore_daily.csv", True, "Conservative timing unless exact release/close timing is documented", "Loaded"])
else:
    acq_rows.append(["IRON_ORE", "source_data/iron_ore_daily.csv", True, "Conservative timing unless exact release/close timing is documented", "Not supplied — excluded"])

for c, s in external.items():
    feat[c] = s

acq = pd.DataFrame(
    acq_rows,
    columns=["Variable", "Source", "Lagged_One_Source_Observation", "Timing_Rationale", "Status"]
)
acq.to_csv(DIAG / "02_market_commodity_acquisition_audit.csv", index=False)
display(acq)

,Variable,Source,Lagged_One_Source_Observation,Timing_Rationale,Status
0,AUDUSD,AUDUSD=X,True,Conservative FX availability at ASX close,Downloaded
1,VIX,^VIX,True,US session timing,Downloaded
2,ASX200,^AXJO,False,Same Australian close is observable,Downloaded
3,GOLD,GC=F,True,Global/US futures timing,Downloaded
4,COPPER,HG=F,True,Global/US futures timing,Downloaded
5,IRON_ORE,source_data/iron_ore_daily.csv,True,Conservative timing unless exact release/close...,Not supplied — excluded


In [6]:
expected_lag_policy = {
    "AUDUSD": True,
    "VIX": True,
    "ASX200": False,
    "GOLD": True,
    "COPPER": True,
}

observed_lag_policy = {
    name: bool(shift_one)
    for name, (_, shift_one, _) in market_map.items()
}

assert observed_lag_policy == expected_lag_policy, (
    "External-market lag policy changed unexpectedly. "
    "Review forecast-origin timing before proceeding."
)

print("External-market timing policy:")
display(
    pd.DataFrame({
        "Variable": list(expected_lag_policy),
        "Lagged_One_Source_Observation": list(expected_lag_policy.values()),
        "Interpretation": [
            "FX observation enters from the next Australian session",
            "US volatility information enters from the next Australian session",
            "Same ASX close is available at forecast origin",
            "Commodity information enters from the next Australian session",
            "Commodity information enters from the next Australian session",
        ],
    })
)

External-market timing policy:


,Variable,Lagged_One_Source_Observation,Interpretation
0,AUDUSD,True,FX observation enters from the next Australian...
1,VIX,True,US volatility information enters from the next...
2,ASX200,False,Same ASX close is available at forecast origin
3,GOLD,True,Commodity information enters from the next Aus...
4,COPPER,True,Commodity information enters from the next Aus...


## 5. F5 Australian macro variables with conservative availability timing


In [7]:
from io import BytesIO
import requests

ABS_API = "https://data.api.abs.gov.au/rest/data"
RBA_F1_URL = "https://www.rba.gov.au/statistics/tables/xls/f01hist.xlsx"
RBA_F2_URL = "https://www.rba.gov.au/statistics/tables/xls/f02hist.xlsx"


def abs_sdmx_csv(dataflow, key, start_period="2016"):
    """Download one filtered ABS SDMX series as CSV (no API key required)."""
    url = (
        f"{ABS_API}/{dataflow}/{key}"
        f"?startPeriod={start_period}&format=csv&dimensionAtObservation=AllDimensions"
    )
    z = pd.read_csv(url)
    if "TIME_PERIOD" not in z.columns or "OBS_VALUE" not in z.columns:
        raise ValueError(f"Unexpected ABS response for {dataflow}: {list(z.columns)}")
    z["OBS_VALUE"] = pd.to_numeric(z["OBS_VALUE"], errors="coerce")
    return z.dropna(subset=["OBS_VALUE"]).copy()


def period_end(series, freq):
    """Convert ABS YYYY-MM / YYYY-Qn labels to the end of the reference period."""
    return pd.PeriodIndex(series.astype(str), freq=freq).to_timestamp(how="end").normalize()


def rba_series_from_table(url, series_id, name):
    """
    Read an official RBA statistical-table workbook and extract a series by its
    stable RBA Series ID rather than by a fragile Excel column letter.
    """
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    raw = pd.read_excel(BytesIO(resp.content), sheet_name=0, header=None)

    id_rows = raw.index[
        raw.apply(lambda row: row.astype(str).str.strip().eq("Series ID").any(), axis=1)
    ].tolist()
    if not id_rows:
        raise ValueError("Could not locate the RBA 'Series ID' metadata row")
    id_row = id_rows[0]

    matches = [j for j, v in enumerate(raw.iloc[id_row]) if str(v).strip() == series_id]
    if not matches:
        raise ValueError(f"RBA series {series_id} not found in {url}")
    value_col = matches[0]

    z = raw.iloc[id_row + 1 :, [0, value_col]].copy()
    z.columns = ["Reference_Date", name]
    z["Reference_Date"] = pd.to_datetime(z["Reference_Date"], errors="coerce")
    z[name] = pd.to_numeric(z[name], errors="coerce")
    return z.dropna().sort_values("Reference_Date")


macro_series = {}
macro_audit = []

# ------------------------------------------------------------
# ABS CPI — quarterly All Groups Australia index -> YoY inflation
# IMPORTANT: ABS changed CPI datasets in Nov-2025 and reinstated a separate CPI_Q
# dataset in Apr-2026. Rather than guessing the new CPI_Q dimension key, this
# notebook uses the exact historical quarterly CPI query documented by ABS:
#   ABS,CPI,1.1.0 / 1.10001.10.50.Q
# This preserves the long quarterly history needed for the 2018+ modelling period.
# Availability: quarter-end + 35 calendar days (conservative approximation).
# ------------------------------------------------------------
try:
    cpi = abs_sdmx_csv("ABS,CPI,1.1.0", "1.10001.10.50.Q", start_period="2016-Q1")
    cpi["Reference_Date"] = period_end(cpi["TIME_PERIOD"], "Q")
    cpi = cpi[["Reference_Date", "OBS_VALUE"]].rename(columns={"OBS_VALUE": "AUS_CPI_INDEX"})
    cpi = cpi.sort_values("Reference_Date")
    cpi["AUS_CPI_YOY"] = 100 * cpi["AUS_CPI_INDEX"].pct_change(4)
    cpi["Available_Date"] = cpi["Reference_Date"] + pd.to_timedelta(35, unit="D")
    macro_series["AUS_CPI_YOY"] = cpi[["Available_Date", "AUS_CPI_YOY"]].dropna()
    macro_audit.append([
        "AUS_CPI_YOY", "ABS", "CPI 1.1.0: 1.10001.10.50.Q", "Quarter-end + 35 days",
        "All Groups Australia CPI index transformed to year-on-year inflation", "Downloaded"
    ])
except Exception as e:
    macro_audit.append([
        "AUS_CPI_YOY", "ABS", "CPI 1.1.0: 1.10001.10.50.Q", "Quarter-end + 35 days",
        "All Groups Australia CPI index transformed to year-on-year inflation",
        f"Unavailable: {type(e).__name__}: {e}"
    ])

# ------------------------------------------------------------
# ABS unemployment — persons, total age, seasonally adjusted, Australia, monthly.
# Exact SDMX key: M13.3.1599.20.AUS.M
# Availability: month-end + 20 calendar days (conservative approximation).
# ------------------------------------------------------------
try:
    unemp = abs_sdmx_csv("LF", "M13.3.1599.20.AUS.M", start_period="2016-01")
    unemp["Reference_Date"] = period_end(unemp["TIME_PERIOD"], "M")
    unemp["AUS_UNEMPLOYMENT"] = pd.to_numeric(unemp["OBS_VALUE"], errors="coerce")
    unemp["Available_Date"] = unemp["Reference_Date"] + pd.to_timedelta(20, unit="D")
    macro_series["AUS_UNEMPLOYMENT"] = unemp[["Available_Date", "AUS_UNEMPLOYMENT"]].dropna()
    macro_audit.append([
        "AUS_UNEMPLOYMENT", "ABS", "LF: M13.3.1599.20.AUS.M", "Month-end + 20 days",
        "Monthly unemployment rate; persons; seasonally adjusted; Australia", "Downloaded"
    ])
except Exception as e:
    macro_audit.append([
        "AUS_UNEMPLOYMENT", "ABS", "LF: M13.3.1599.20.AUS.M", "Month-end + 20 days",
        "Monthly unemployment rate; persons; seasonally adjusted; Australia",
        f"Unavailable: {type(e).__name__}: {e}"
    ])

# ------------------------------------------------------------
# ABS real GDP — seasonally adjusted chain-volume GDP -> QoQ growth.
# Exact SDMX key: M1.GPM.20.AUS.Q
# Availability: quarter-end + 70 calendar days. This is deliberately conservative
# relative to the normal quarterly National Accounts publication timetable.
# ------------------------------------------------------------
try:
    gdp = abs_sdmx_csv("ANA_AGG", "M1.GPM.20.AUS.Q", start_period="2016-Q1")
    gdp["Reference_Date"] = period_end(gdp["TIME_PERIOD"], "Q")
    gdp["AUS_REAL_GDP_LEVEL"] = pd.to_numeric(gdp["OBS_VALUE"], errors="coerce")
    gdp = gdp.sort_values("Reference_Date")
    gdp["AUS_GDP_QOQ"] = 100 * gdp["AUS_REAL_GDP_LEVEL"].pct_change()
    gdp["Available_Date"] = gdp["Reference_Date"] + pd.to_timedelta(70, unit="D")
    macro_series["AUS_GDP_QOQ"] = gdp[["Available_Date", "AUS_GDP_QOQ"]].dropna()
    macro_audit.append([
        "AUS_GDP_QOQ", "ABS", "ANA_AGG: M1.GPM.20.AUS.Q", "Quarter-end + 70 days",
        "Seasonally adjusted chain-volume real GDP transformed to quarter-on-quarter growth", "Downloaded"
    ])
except Exception as e:
    macro_audit.append([
        "AUS_GDP_QOQ", "ABS", "ANA_AGG: M1.GPM.20.AUS.Q", "Quarter-end + 70 days",
        "Seasonally adjusted chain-volume real GDP transformed to quarter-on-quarter growth",
        f"Unavailable: {type(e).__name__}: {e}"
    ])

# ------------------------------------------------------------
# RBA short rate — F1.1, 90-day BABs/NCDs, Series ID FIRMMBAB90.
# Monthly RBA table observations are aligned conservatively at month-end + 10 days.
# ------------------------------------------------------------
try:
    short = rba_series_from_table(RBA_F1_URL, "FIRMMBAB90", "AUS_SHORT_RATE")
    short["Reference_Date"] = short["Reference_Date"].dt.to_period("M").dt.to_timestamp("M")
    short["Available_Date"] = short["Reference_Date"] + pd.to_timedelta(10, unit="D")
    macro_series["AUS_SHORT_RATE"] = short[["Available_Date", "AUS_SHORT_RATE"]].dropna()
    macro_audit.append([
        "AUS_SHORT_RATE", "RBA", "F1.1 / FIRMMBAB90", "Month-end + 10 days",
        "Monthly 90-day Bank Accepted Bills / Negotiable Certificates of Deposit rate", "Downloaded"
    ])
except Exception as e:
    macro_audit.append([
        "AUS_SHORT_RATE", "RBA", "F1.1 / FIRMMBAB90", "Month-end + 10 days",
        "Monthly 90-day Bank Accepted Bills / Negotiable Certificates of Deposit rate",
        f"Unavailable: {type(e).__name__}: {e}"
    ])

# ------------------------------------------------------------
# RBA long rate — F2.1, Australian Government 10-year bond,
# Series ID FCMYGBAG10. Same conservative monthly availability rule.
# ------------------------------------------------------------
try:
    longr = rba_series_from_table(RBA_F2_URL, "FCMYGBAG10", "AUS_LONG_RATE")
    longr["Reference_Date"] = longr["Reference_Date"].dt.to_period("M").dt.to_timestamp("M")
    longr["Available_Date"] = longr["Reference_Date"] + pd.to_timedelta(10, unit="D")
    macro_series["AUS_LONG_RATE"] = longr[["Available_Date", "AUS_LONG_RATE"]].dropna()
    macro_audit.append([
        "AUS_LONG_RATE", "RBA", "F2.1 / FCMYGBAG10", "Month-end + 10 days",
        "Monthly Australian Government 10-year bond yield", "Downloaded"
    ])
except Exception as e:
    macro_audit.append([
        "AUS_LONG_RATE", "RBA", "F2.1 / FCMYGBAG10", "Month-end + 10 days",
        "Monthly Australian Government 10-year bond yield",
        f"Unavailable: {type(e).__name__}: {e}"
    ])

# ------------------------------------------------------------
# Merge every macro series independently by its AVAILABLE date.
# This preserves the most recently available observation for each variable even
# when other macro series update on different dates.
# ------------------------------------------------------------
base_dates = pd.DataFrame({"Date": feat.index}).sort_values("Date")

for name, z in macro_series.items():
    zz = (
        z.replace([np.inf, -np.inf], np.nan)
         .dropna(subset=["Available_Date", name])
         .sort_values("Available_Date")
         .drop_duplicates("Available_Date", keep="last")
         .rename(columns={"Available_Date": "Date"})
    )
    aligned = pd.merge_asof(
        base_dates,
        zz[["Date", name]],
        on="Date",
        direction="backward",
        allow_exact_matches=True
    ).set_index("Date")[name]
    feat[name] = aligned.reindex(feat.index)

if {"AUS_LONG_RATE", "AUS_SHORT_RATE"}.issubset(feat.columns):
    feat["AUS_YIELD_SPREAD"] = feat["AUS_LONG_RATE"] - feat["AUS_SHORT_RATE"]

_failed_macro = [row for row in macro_audit if row[-1] != "Downloaded"]
if _failed_macro:
    _msg = "\n".join(f"  - {row[0]}: {row[-1]}" for row in _failed_macro)
    raise RuntimeError(
        "One or more required F5 macro series could not be acquired:\n" + _msg +
        "\nFix the source query before continuing so F5 remains reproducible."
    )

macro_audit = pd.DataFrame(
    macro_audit,
    columns=[
        "Feature", "Source", "Source_Series", "Availability_Method",
        "Description", "Status"
    ]
)
macro_audit.to_csv(DIAG / "02_macro_timing_audit.csv", index=False)
display(macro_audit)

macro_features_expected = [
    "AUS_CPI_YOY", "AUS_UNEMPLOYMENT", "AUS_GDP_QOQ",
    "AUS_SHORT_RATE", "AUS_LONG_RATE", "AUS_YIELD_SPREAD"
]
print("Macro features present:")
print([c for c in macro_features_expected if c in feat.columns])


,Feature,Source,Source_Series,Availability_Method,Description,Status
0,AUS_CPI_YOY,ABS,CPI 1.1.0: 1.10001.10.50.Q,Quarter-end + 35 days,All Groups Australia CPI index transformed to ...,Downloaded
1,AUS_UNEMPLOYMENT,ABS,LF: M13.3.1599.20.AUS.M,Month-end + 20 days,Monthly unemployment rate; persons; seasonally...,Downloaded
2,AUS_GDP_QOQ,ABS,ANA_AGG: M1.GPM.20.AUS.Q,Quarter-end + 70 days,Seasonally adjusted chain-volume real GDP tran...,Downloaded
3,AUS_SHORT_RATE,RBA,F1.1 / FIRMMBAB90,Month-end + 10 days,Monthly 90-day Bank Accepted Bills / Negotiabl...,Downloaded
4,AUS_LONG_RATE,RBA,F2.1 / FCMYGBAG10,Month-end + 10 days,Monthly Australian Government 10-year bond yield,Downloaded


Macro features present:
['AUS_CPI_YOY', 'AUS_UNEMPLOYMENT', 'AUS_GDP_QOQ', 'AUS_SHORT_RATE', 'AUS_LONG_RATE', 'AUS_YIELD_SPREAD']


## 6. Locked nested feature groups and processed horizon datasets


In [8]:
feat = feat.replace([np.inf, -np.inf], np.nan)

F1 = [c for c in feat.columns if c.startswith(("RV_", "EWMA_", "RV21_", "VolOfVol_"))]

F2 = list(dict.fromkeys(
    F1 + [
        c for c in feat.columns
        if c.startswith(("Return_", "Abs_Return", "Squared_Return", "Downside_", "Drawdown_"))
    ]
))

F3 = list(dict.fromkeys(
    F2 + [
        c for c in feat.columns
        if c.startswith(("Sector_", "Dollar_"))
    ]
))

f4_prefixes = ("AUDUSD_", "VIX_", "ASX200_", "GOLD_", "COPPER_", "IRON_ORE_")
F4 = list(dict.fromkeys(
    F3 + [c for c in feat.columns if c.startswith(f4_prefixes)]
))

macro_names = [
    c for c in [
        "AUS_CPI_YOY",
        "AUS_UNEMPLOYMENT",
        "AUS_LONG_RATE",
        "AUS_SHORT_RATE",
        "AUS_GDP_QOQ",
        "AUS_YIELD_SPREAD",
    ]
    if c in feat.columns
]
F5 = list(dict.fromkeys(F4 + macro_names))

groups = {
    "F1_Volatility": F1,
    "F2_Returns_Downside": F2,
    "F3_Range_Liquidity": F3,
    "F4_Market_Commodities": F4,
    "F5_Plus_Macro": F5,
}

rows = []
previous = []
for name, cols in groups.items():
    added = [c for c in cols if c not in previous]
    rows.append({
        "Feature_Set": name,
        "N_Features": len(cols),
        "Added": ", ".join(added)
    })
    previous = cols

group_def = pd.DataFrame(rows)
group_def.to_csv(TABLES / "02_feature_group_definition.csv", index=False)
display(group_def)

full = (
    feat.reset_index()
    .merge(target, on="Date", how="left")
    .merge(sector[["Date", "Sector_Return"]], on="Date", how="left")
)

# Explicit leakage guard.
for fs, cols in groups.items():
    bad = [c for c in cols if c.startswith("Target_RV_")]
    assert not bad, f"Target leakage in {fs}: {bad}"

for h in HORIZONS:
    y = f"Target_RV_{h}"
    d = full[["Date", "Sector_Return", y] + F5].copy()
    d = d.dropna(subset=[y])
    d.to_parquet(PROCESSED / f"volatility_features_h{h}.parquet", index=False)
    d.to_csv(PROCESSED / f"volatility_features_h{h}.csv", index=False)

print("Saved horizon datasets:", HORIZONS)

,Feature_Set,N_Features,Added
0,F1_Volatility,10,"RV_5, RV_10, RV_21, RV_63, RV_126, EWMA_Vol_09..."
1,F2_Returns_Downside,21,"Return_1D, Abs_Return_1D, Squared_Return_1D, R..."
2,F3_Range_Liquidity,26,"Sector_HL_Range, Sector_Parkinson_Var, Sector_..."
3,F4_Market_Commodities,42,"AUDUSD_Return_1D, AUDUSD_Return_21D, AUDUSD_RV..."
4,F5_Plus_Macro,48,"AUS_CPI_YOY, AUS_UNEMPLOYMENT, AUS_LONG_RATE, ..."


Saved horizon datasets: [5, 21, 63]


## 7. Missingness, timing and target-correlation diagnostics


In [9]:
miss_rows = []
for c in F5:
    for label, mask in [
        ("2018_plus", full["Date"] >= "2018-01-01"),
        ("2023_plus", full["Date"] >= "2023-01-01"),
        ("2025_plus", full["Date"] >= "2025-01-01"),
    ]:
        x = full.loc[mask, c]
        miss_rows.append({
            "Feature": c,
            "Period": label,
            "N": len(x),
            "Missing_N": int(x.isna().sum()),
            "Missing_Pct": 100 * x.isna().mean() if len(x) else np.nan,
            "First_Valid_Date": full.loc[mask & full[c].notna(), "Date"].min()
        })

missingness = pd.DataFrame(miss_rows)
missingness.to_csv(DIAG / "02_feature_missingness_by_period.csv", index=False)

for h in HORIZONS:
    y = f"Target_RV_{h}"
    corr = (
        full[[y] + F5]
        .corr(numeric_only=True)[y]
        .drop(y)
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )
    corr.rename("Correlation").to_csv(TABLES / f"02_target_correlations_h{h}.csv")

display(pd.read_csv(TABLES / "02_target_correlations_h21.csv").head(25))
display(missingness[missingness["Period"].eq("2025_plus")].sort_values("Missing_Pct", ascending=False).head(25))

,Unnamed: 0,Correlation
0,RV_21,0.862608
1,EWMA_Vol_094,0.854254
2,Downside_Vol_21,0.851463
3,RV_10,0.844785
4,RV_5,0.802247
5,Sector_HL_Range,0.792064
6,RV_63,0.788836
7,Downside_Vol_63,0.786588
8,Sector_Parkinson_Vol,0.782286
9,Sector_Parkinson_Var,0.745738


,Feature,Period,N,Missing_N,Missing_Pct,First_Valid_Date
2,RV_5,2025_plus,435,0,0.0,2025-01-02
5,RV_10,2025_plus,435,0,0.0,2025-01-02
80,AUDUSD_Return_1D,2025_plus,435,0,0.0,2025-01-02
83,AUDUSD_Return_21D,2025_plus,435,0,0.0,2025-01-02
86,AUDUSD_RV21,2025_plus,435,0,0.0,2025-01-02
89,VIX_Return_1D,2025_plus,435,0,0.0,2025-01-02
92,VIX_Return_21D,2025_plus,435,0,0.0,2025-01-02
95,VIX_RV21,2025_plus,435,0,0.0,2025-01-02
98,VIX_Level,2025_plus,435,0,0.0,2025-01-02
101,ASX200_Return_1D,2025_plus,435,0,0.0,2025-01-02
